# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Muhammadfaheem8988/FlyRank-AI-internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## Step 0: Setup and Environment

In [1]:
import os
import duckdb
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.metrics import roc_auc_score, average_precision_score, precision_score
from sklearn.ensemble import HistGradientBoostingClassifier

# DuckDB Setup & HF Secret
con = duckdb.connect()

try:
    from google.colab import userdata
    hf_token = userdata.get('HF_TOKEN')
except Exception:
    hf_token = os.environ.get('HF_TOKEN', '')

if hf_token:
    con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{hf_token}')")
    print("DuckDB Hugging Face secret configured.")
else:
    print("Warning: HF_TOKEN not found.")

WAREHOUSE = "hf://datasets/FlyRank/internship-warehouse"
FACT_DAILY = f"{WAREHOUSE}/fact_content_daily_performance"

# Extract feature observation month (2026-03) and ground-truth outcome month (2026-04)
df = con.execute(f"""
    WITH march_obs AS (
        SELECT
            content_hash_id,
            ANY_VALUE(client_hash_id) AS client_hash_id,
            SUM(gsc_clicks) AS clicks,
            SUM(gsc_impressions) AS impressions,
            SUM(gsc_sum_position) / NULLIF(SUM(gsc_impressions), 0) AS avg_position,
            SUM(gsc_clicks)::FLOAT / NULLIF(SUM(gsc_impressions), 0) AS ctr,
            SUM(sessions_organic) AS organic_sessions,
            SUM(ga4_total_engagement_sec)::FLOAT / NULLIF(SUM(sessions_organic), 0) AS avg_engagement_sec,
            COUNT(DISTINCT report_date) AS active_days
        FROM read_parquet('{FACT_DAILY}/month=2026-03/*.parquet')
        WHERE gsc_data_available IS TRUE
        GROUP BY content_hash_id
        HAVING SUM(gsc_impressions) >= 10 IS TRUE
    ),
    april_outcome AS (
        SELECT
            content_hash_id,
            SUM(gsc_clicks) AS april_clicks
        FROM read_parquet('{FACT_DAILY}/month=2026-04/*.parquet')
        WHERE gsc_data_available IS TRUE
        GROUP BY content_hash_id
    )
    SELECT
        m.*,
        COALESCE(a.april_clicks, 0) AS april_clicks,
        CASE WHEN COALESCE(a.april_clicks, 0) > m.clicks THEN 1 ELSE 0 END AS target_rebound
    FROM march_obs m
    LEFT JOIN april_outcome a ON m.content_hash_id = a.content_hash_id
""").df().fillna(0)

# Expected CTR curve for CTR gap feature
def expected_ctr(pos):
    if pos <= 3: return 0.20
    elif pos <= 6: return 0.08
    elif pos <= 10: return 0.03
    elif pos <= 20: return 0.015
    else: return 0.005

df['expected_ctr'] = df['avg_position'].apply(expected_ctr)
df['ctr_gap'] = (df['expected_ctr'] - df['ctr']).clip(lower=0)

features = [
    'clicks',
    'impressions',
    'avg_position',
    'ctr',
    'organic_sessions',
    'avg_engagement_sec',
    'active_days',
    'ctr_gap'
]

print(f"Loaded dataset: {len(df):,} items across {df['client_hash_id'].nunique()} unique clients.")

DuckDB Hugging Face secret configured.


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Loaded dataset: 143,206 items across 45 unique clients.


## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

### Constructive Methodological Review of the Research Paper

Reviewing published literature with an ML engineering lens requires asking constructive questions regarding data provenance, label derivation, and evaluation integrity:

#### Paper Finding 1: Refresh Intervention Recovery Rates
* **Claim in Paper**: Content updates and metadata optimizations lead to an observed lift in average ranking position and organic traffic recovery within 30 to 60 days post-refresh.
* **Methodological Question**: *How was the intervention label isolated from concurrent macro shifts, and what was the counterfactual?*
  * In non-randomized observational web data, content items chosen by human editors for refreshing are systematically different from neglected items (selection bias).
  * Without an untreated control group or synthetic difference-in-differences design, how much of the measured 30–60 day trajectory represents genuine intervention impact versus mean reversion or broad seasonality?

#### Paper Finding 2: Cross-Domain Model Portability
* **Claim in Paper**: Predictive prioritization models trained on historical search console signals demonstrate strong cross-client ranking utility.
* **Methodological Question**: *Does the cross-client validation partition prevent latent domain-level information leakage?*
  * If client domains with high shared link equity, shared parent CMS architectures, or identical syndication networks appear across both training and validation splits, validation metrics will be artificially optimistic.
  * We ask whether the validation design strictly enforced grouped out-of-domain partitions at the parent entity level and held out time-forward observation slices.

In [2]:
# Verify class balance and client distribution relevant to the methodological critique
client_summary = df.groupby('client_hash_id').agg(
    n_urls=('content_hash_id', 'count'),
    mean_clicks=('clicks', 'mean'),
    rebound_rate=('target_rebound', 'mean')
).reset_index()

print("--- Client Cohort Heterogeneity (Demonstrating domain-level variance) ---")
print(client_summary.describe().round(3))

--- Client Cohort Heterogeneity (Demonstrating domain-level variance) ---
          n_urls  mean_clicks  rebound_rate
count     45.000       45.000        45.000
mean    3182.356        2.411         0.150
std     5863.704        3.538         0.154
min        1.000        0.000         0.000
25%       75.000        0.333         0.044
50%      944.000        0.904         0.111
75%     3152.000        2.690         0.222
max    26355.000       14.636         0.882


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

### Honest Split Audit: Random Split vs. Client-Grouped Split

* **The Leakage Threat in Naive Random Splitting**: In an unstratified random row split, individual URLs from the *same* client domain appear simultaneously in train and test sets. The model can exploit client-level base rates, domain authority, and sitewide template structures rather than learning generalized content-level signals.
* **The Honest Grouped Design**: We implement a `GroupShuffleSplit` partitioned on `client_hash_id` (75% train clients, 25% holdout clients). The model is tested solely on entirely unseen web properties.
* **Measured Impact**: We observe a realistic drop in performance metrics when moving from the naive split to the honest grouped split, reflecting true out-of-domain generalization without optimistic bias.

In [3]:
# 1. NAIVE RANDOM SPLIT (Row-level random split)
X_rnd_train, X_rnd_test, y_rnd_train, y_rnd_test = train_test_split(
    df[features], df['target_rebound'], test_size=0.25, random_state=42
)

model_naive = HistGradientBoostingClassifier(max_iter=150, learning_rate=0.08, random_state=42)
model_naive.fit(X_rnd_train, y_rnd_train)
preds_naive = model_naive.predict_proba(X_rnd_test)[:, 1]

auc_naive = roc_auc_score(y_rnd_test, preds_naive)
prauc_naive = average_precision_score(y_rnd_test, preds_naive)

# 2. HONEST GROUPED SPLIT (Client-level holdout)
gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(gss.split(df, groups=df['client_hash_id']))

train_grp = df.iloc[train_idx].copy()
test_grp = df.iloc[test_idx].copy()

X_grp_train, y_grp_train = train_grp[features], train_grp['target_rebound']
X_grp_test, y_grp_test = test_grp[features], test_grp['target_rebound']

model_honest = HistGradientBoostingClassifier(max_iter=150, learning_rate=0.08, random_state=42)
model_honest.fit(X_grp_train, y_grp_train)
preds_honest = model_honest.predict_proba(X_grp_test)[:, 1]

auc_honest = roc_auc_score(y_grp_test, preds_honest)
prauc_honest = average_precision_score(y_grp_test, preds_honest)

# Compile Before/After Comparison Table
split_comparison = pd.DataFrame({
    'Split Design': ['Naive Random Split (Leaky)', 'Client-Grouped Split (Honest)'],
    'Train Size': [f"{len(X_rnd_train):,}", f"{len(X_grp_train):,} ({train_grp['client_hash_id'].nunique()} clients)"],
    'Test Size': [f"{len(X_rnd_test):,}", f"{len(X_grp_test):,} ({test_grp['client_hash_id'].nunique()} clients)"],
    'ROC-AUC': [round(auc_naive, 4), round(auc_honest, 4)],
    'PR-AUC': [round(prauc_naive, 4), round(prauc_honest, 4)],
    'Delta (Honest - Naive)': ['Baseline', f"{auc_honest - auc_naive:+.4f} AUC"]
})

print("--- Before vs. After: Split Design Validation Audit ---")
display(split_comparison)

--- Before vs. After: Split Design Validation Audit ---


,Split Design,Train Size,Test Size,ROC-AUC,PR-AUC,Delta (Honest - Naive)
0,Naive Random Split (Leaky),"107,404","35,802",0.7458,0.4539,Baseline
1,Client-Grouped Split (Honest),"110,318 (33 clients)","32,888 (12 clients)",0.6704,0.3055,-0.0754 AUC


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

### 3. Leakage Audit & Failure Mode Inspection

#### 1. Temporal Boundary & Contamination Audit
* **Temporal Cutoff**: All eight features (`clicks`, `impressions`, `avg_position`, `ctr`, `organic_sessions`, `avg_engagement_sec`, `active_days`, `ctr_gap`) were computed exclusively from March 2026 logs (`report_date <= 2026-03-31`).
* **Target Isolation**: The label `target_rebound` and the comparison metric `april_clicks` were calculated strictly from the April 2026 window (`month=2026-04`).
* **Correlation Verification**: No feature displays an artificially high correlation ($r \approx 1.0$) with `target_rebound`, confirming no direct target proxies leaked into the training matrix.

#### 2. Concrete Failure Mode Inspection

* **High-Confidence False Positives (Predicted Rebound, Actual: Stagnant)**:
  * `content_1db634b4093a38eb`: Model assigned a **95.2%** rebound probability (50 impressions, position 4.90, 1 click), but April clicks remained flat at **1.0**.
  * `content_b0fb7d22899634c2`: Model assigned a **92.0%** rebound probability (33 impressions, top rank position 0.48), but April clicks fell to **0.0**.
  * *Diagnosis*: Low-sample volatility on striking-distance queries. When impression counts are under 100, ranking metrics produce noisy feature combinations that fool non-linear splits.

* **High-Confidence False Negatives (Predicted Stagnant, Actual: Rebounded)**:
  * `content_b8eb55e9605771a2`: Model predicted only **1.1%** rebound chance due to poor rank (position 68.75), but April clicks surged from **3.0 to 7.0**.
  * `content_d5b712921c05df84` & `content_e5ae0f3c0c64adb0`: Model predicted **1.2%** probability (positions ~80–84, 0 clicks), yet both gained their first click in April.
  * *Diagnosis*: Deep-tail discovery and indexing lag. Pages ranking on Page 7–9 can jump suddenly onto Page 2–3 via external backlink acquisition or competitor URL deprecation—unmeasured signals absent from March GSC logs.

In [6]:
# 1. Feature Correlation with Target vs Leaked Proxy
correlations = df[features + ['april_clicks', 'target_rebound']].corr()['target_rebound'].sort_values(ascending=False)
print("--- Linear Correlation with Target ---")
print(correlations.round(4))

# 2. Extract concrete False Positives and False Negatives
test_grp['predicted_prob'] = preds_honest
test_grp['error'] = test_grp['predicted_prob'] - test_grp['target_rebound']

top_fp = test_grp[test_grp['target_rebound'] == 0].sort_values(by='predicted_prob', ascending=False).head(3)
top_fn = test_grp[test_grp['target_rebound'] == 1].sort_values(by='predicted_prob', ascending=True).head(3)

print("\n--- Failure Mode Example: Top False Positives (Predicted Rebound, Actual: Stagnated) ---")
display(top_fp[['content_hash_id', 'predicted_prob', 'impressions', 'clicks', 'avg_position', 'ctr', 'april_clicks']])

print("\n--- Failure Mode Example: Top False Negatives (Predicted Stagnant, Actual: Surged) ---")
display(top_fn[['content_hash_id', 'predicted_prob', 'impressions', 'clicks', 'avg_position', 'ctr', 'april_clicks']])

--- Linear Correlation with Target ---
target_rebound        1.0000
april_clicks          0.1807
impressions           0.0919
clicks                0.0693
active_days           0.0687
organic_sessions      0.0654
ctr_gap               0.0594
avg_engagement_sec    0.0252
ctr                   0.0056
avg_position         -0.1161
Name: target_rebound, dtype: float64

--- Failure Mode Example: Top False Positives (Predicted Rebound, Actual: Stagnated) ---


,content_hash_id,predicted_prob,impressions,clicks,avg_position,ctr,april_clicks
69027,content_1db634b4093a38eb,0.952049,50.0,1.0,4.900000,0.020000,1.0
56240,content_b0fb7d22899634c2,0.920464,33.0,1.0,0.484848,0.030303,0.0
61439,content_4fbb90291812e7be,0.873588,18.0,1.0,5.222222,0.055556,1.0



--- Failure Mode Example: Top False Negatives (Predicted Stagnant, Actual: Surged) ---


,content_hash_id,predicted_prob,impressions,clicks,avg_position,ctr,april_clicks
50139,content_b8eb55e9605771a2,0.011040,80.0,3.0,68.750000,0.0375,7.0
32431,content_d5b712921c05df84,0.011524,43.0,0.0,84.372093,0.0000,1.0
48174,content_e5ae0f3c0c64adb0,0.012142,31.0,0.0,79.967742,0.0000,1.0


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

### 4. Claim Rewrite: Defending Trustworthy Statements

To ensure our reporting adheres to engineering rigor and public-safe language, we audit past assertions and rewrite them using measured terms (*observed*, *measured*, *directional*, *decision-support*):

| Prior / Overstated Claim | Audited & Rewritten Safe Claim | Methodological Reason for Rewrite |
|---|---|---|
| *"Our machine learning model accurately predicts which content will recover next month with ~75% AUC."* | *"In a naive row split, the model reached 0.7458 ROC-AUC, but under an honest client-grouped partition (holding out 12 unseen client domains), the measured ROC-AUC is 0.6704 (PR-AUC 0.3055). This provides directional decision-support rather than guaranteed recovery prediction."* | Removes optimistic domain leakage bias; replaces inflated 0.75 figure with the honest 0.6704 out-of-domain benchmark. |
| *"The model proves that ranking in striking distance with CTR gap causes content traffic rebound."* | *"We observed that striking-distance position and CTR deficit correlate with higher probability of rebound, but observational logs cannot prove causation or rule out concurrent SERP feature volatility."* | Replaces causal overreach ("proves", "causes") with observed statistical correlation. |
| *"Deploying this model eliminates editorial wasted effort across client portfolios."* | *"The model serves as a prioritization heuristic to filter candidates under editorial capacity constraints, though top-ranked candidates still exhibit failure modes on low-impression tail queries and zero-click SERP layouts."* | Replaces deterministic operational promise with measured decision-support boundaries and failure mode disclosure. |

In [7]:
# Print exact receipts for Section 4 claim rewrite
print(f"Naive Split Test ROC-AUC:  {auc_naive:.4f}")
print(f"Honest Split Test ROC-AUC: {auc_honest:.4f} (Delta: {auc_honest - auc_naive:.4f})")
print(f"Honest Split Test PR-AUC:  {prauc_honest:.4f}")
print(f"Test Set Composition:      {len(test_grp):,} URLs across {test_grp['client_hash_id'].nunique()} unseen client domains")

Naive Split Test ROC-AUC:  0.7458
Honest Split Test ROC-AUC: 0.6704 (Delta: -0.0754)
Honest Split Test PR-AUC:  0.3055
Test Set Composition:      32,888 URLs across 12 unseen client domains


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.